# Part 2: Query structured data with Fabric IQ

Part 2 at Caldova begins when a procurement analyst finds Meridian API Works invoice and quality evidence for lot `AUR-API-7`. The lot contains Caldovexine (`SUB-001`). The analyst needs two different kinds of structured answers. First, the business **relationships**: which medicinal products contain the substance and which manufacturers make it. Second, the **performance trend**: is Meridian API Works getting better or worse at delivering on time and in full? Fabric IQ answers both from Caldova's lakehouse.

**🎯 Mission**
- Query the Caldova **ontology** through its MCP endpoint for product, substance, and manufacturer relationships
- Create and publish a **Fabric data agent** over the supplier analytics tables in the lakehouse
- Ask the data agent analytical questions through its MCP endpoint and compare the two kinds of answers

**Before you run:** Follow [Exercise 2 in the lab guide](../../instructions/overview.md#exercise-2-query-structured-data-with-fabric-iq) to explore the lakehouse tables and the ontology in Microsoft Fabric. Then follow this notebook from top to bottom, running the code cells in sequence.

## The building blocks

- **Fabric IQ ontology**: Gives business meaning to lakehouse tables as entities (such as `MedicinalProduct` and `ActiveSubstance`) and relationships (such as **contains** and **manufactures**). Its MCP endpoint exposes an `ask_ontology` tool that answers questions over those relationships.
- **Fabric data agent**: Turns natural-language questions into queries over the tables you select. Once published, it exposes an MCP endpoint with a tool named after the agent.
- **Delegated sign-in**: Both endpoints run as you. The notebook signs you in with `InteractiveBrowserCredential` and uses your Fabric permissions.

## Step 1: Load configuration and sign in to Fabric

Load the Fabric workspace and ontology settings written to `.env` during setup, sign in with your lab account, and define a helper that opens an authenticated MCP session for any Fabric MCP URL. The HTTP client follows redirects because Fabric routes MCP requests through its service endpoint.

If the browser sign-in window is hidden, minimize other windows **inside the virtual machine**. Keep the outer browser hosting Skillable open.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

FABRIC_TENANT_ID = os.getenv("FABRIC_TENANT_ID") or os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]
FABRIC_ONTOLOGY_MCP_URL = os.getenv("FABRIC_ONTOLOGY_MCP_URL") or (
    "https://api.fabric.microsoft.com/v1/mcp/dataPlane/"
    f"workspaces/{FABRIC_WORKSPACE_ID}/items/{FABRIC_ONTOLOGY_ID}/ontologyEndpoint"
)
FABRIC_API_URL = "https://api.fabric.microsoft.com"
FABRIC_SCOPE = f"{FABRIC_API_URL}/.default"

credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)
credential.get_token(FABRIC_SCOPE)


@asynccontextmanager
async def open_fabric_mcp_session(mcp_url):
    token = credential.get_token(FABRIC_SCOPE)
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        follow_redirects=True,
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(mcp_url, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session


print("Signed in to Fabric")

---

## A: Query the ontology through MCP

## Step 2: List the ontology tools

Open an authenticated MCP session to the `CaldovaMedicinalProductOntology` endpoint and inspect every tool it exposes, including each tool's input schema.

In [ ]:
from rich.console import Console

console = Console()

async with open_fabric_mcp_session(FABRIC_ONTOLOGY_MCP_URL) as session:
    tools_response = await session.list_tools()

for tool in tools_response.tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

## Step 3: Ask the ontology about Caldovexine

Call the ontology's `ask_ontology` tool with a relationship question. The tool answers from the lakehouse data bound to the ontology and returns the answer with supporting evidence. Compare the answer with the `ProductActiveSubstance` and `ManufacturerActiveSubstance` rows you previewed in Fabric.

⏱️ Answers typically take 30 to 60 seconds. If a question takes longer than about a minute, Fabric can return an empty response, which surfaces as `Error parsing JSON response` followed by a timeout. If that happens, rerun the cell.

In [ ]:
from datetime import timedelta

from rich.markdown import Markdown

question = "Which manufacturers make Caldovexine (SUB-001), and which Caldova medicinal products contain it?"

async with open_fabric_mcp_session(FABRIC_ONTOLOGY_MCP_URL) as session:
    result = await session.call_tool(
        "ask_ontology",
        {"request": question},
        read_timeout_seconds=timedelta(seconds=120),
    )

if result.isError:
    raise RuntimeError(f"ask_ontology failed: {result.content}")
ontology_answer = "".join(block.text for block in result.content if block.type == "text")
console.print(Markdown(ontology_answer))

---

## B: Create a Fabric data agent and query it through MCP

The ontology answers relationship questions. Trend questions, such as how a supplier's on-time-in-full (OTIF) rate changed week by week, need aggregation over the `SupplierPerformance` table. A **Fabric data agent** handles that kind of question. You create one with the Fabric REST API in the next three steps.

## Step 4: Find the lakehouse and create the data agent

Define a small helper for Fabric REST calls that waits for long-running operations. Then look up the `CaldovaSupplierAnalytics` lakehouse and create the `CaldovaSupplierDataAgent` item in your workspace. If the data agent already exists, for example because you reran this cell, the notebook reuses it.

In [ ]:
import time

LAKEHOUSE_NAME = "CaldovaSupplierAnalytics"
DATA_AGENT_NAME = "CaldovaSupplierDataAgent"
WORKSPACE_URL = f"/v1/workspaces/{FABRIC_WORKSPACE_ID}"

fabric = httpx.Client(
    base_url=FABRIC_API_URL,
    headers={"Authorization": f"Bearer {credential.get_token(FABRIC_SCOPE).token}"},
    timeout=60,
)


def fabric_call(method, url, **kwargs):
    """Send a Fabric REST request and wait for a long-running operation to finish."""
    response = fabric.request(method, url, **kwargs)
    response.raise_for_status()
    if response.status_code == 202 and "Location" in response.headers:
        for _ in range(150):
            operation = fabric.get(response.headers["Location"])
            operation.raise_for_status()
            status = operation.json().get("status")
            if status == "Succeeded":
                break
            if status in ("Failed", "Cancelled"):
                raise RuntimeError(f"Fabric operation {status.lower()}: {operation.json().get('error')}")
            time.sleep(2)
        else:
            raise TimeoutError(f"Timed out waiting for {method} {url}")
    return response


def list_values(url, **params):
    """Return every item from a paged Fabric list response."""
    page = fabric_call("GET", url, params=params).json()
    values = list(page.get("value", []))
    while page.get("continuationUri"):
        page = fabric_call("GET", page["continuationUri"]).json()
        values.extend(page.get("value", []))
    return values


def find_by_name(items, name):
    return next((item for item in items if item.get("displayName") == name), None)


lakehouse = find_by_name(list_values(f"{WORKSPACE_URL}/lakehouses"), LAKEHOUSE_NAME)
if lakehouse is None:
    raise RuntimeError(f"Lakehouse '{LAKEHOUSE_NAME}' was not found in workspace {FABRIC_WORKSPACE_ID}.")

data_agent = find_by_name(list_values(f"{WORKSPACE_URL}/dataAgents"), DATA_AGENT_NAME)
if data_agent is None:
    fabric_call("POST", f"{WORKSPACE_URL}/dataAgents", json={"displayName": DATA_AGENT_NAME})
    for _ in range(30):
        data_agent = find_by_name(list_values(f"{WORKSPACE_URL}/dataAgents"), DATA_AGENT_NAME)
        if data_agent:
            break
        time.sleep(2)
    else:
        raise RuntimeError(f"Data agent '{DATA_AGENT_NAME}' was created but could not be found.")
    print(f"Created data agent '{DATA_AGENT_NAME}'")
else:
    print(f"Reusing data agent '{DATA_AGENT_NAME}'")

lakehouse_id = lakehouse["id"]
data_agent_id = data_agent["id"]
DATA_AGENT_URL = f"{WORKSPACE_URL}/dataAgents/{data_agent_id}"
print(f"Lakehouse ID:  {lakehouse_id}")
print(f"Data agent ID: {data_agent_id}")

## Step 5: Add the lakehouse and select the supplier analytics tables

Add the lakehouse as a data source of the data agent's draft (staging) configuration, then choose which tables it may query. Select the supplier analytics star schema: the `SupplierPerformance` weekly snapshots and the dimension tables that describe suppliers, dates, locations, and ratings. The ontology already covers product and substance relationships, so those tables stay unselected.

The elements API returns the lakehouse as a tree of schemas, tables, and columns. `recursive=true` returns the whole tree in one call.

In [ ]:
SUPPLIER_ANALYTICS_TABLES = {
    "SupplierPerformance",
    "DimSupplier",
    "DimDate",
    "DimLocation",
    "DimInspectionResult",
    "DimAuditResult",
    "DimFinancialRating",
}


def find_datasource():
    return next(
        (
            source
            for source in list_values(f"{DATA_AGENT_URL}/staging/datasources")
            if source.get("itemReference", {}).get("itemId") == lakehouse_id
        ),
        None,
    )


datasource = find_datasource()
if datasource is None:
    fabric_call(
        "POST",
        f"{DATA_AGENT_URL}/staging/datasources",
        json={
            "type": "FabricItem",
            "itemReference": {"referenceType": "ById", "itemId": lakehouse_id, "workspaceId": FABRIC_WORKSPACE_ID},
        },
    )
    datasource = find_datasource()
datasource_id = datasource["id"]
elements_url = f"{DATA_AGENT_URL}/staging/datasources/{datasource_id}/elements"

tables = {element["displayName"]: element for element in list_values(elements_url, recursive="true") if element["type"] == "Table"}
missing = SUPPLIER_ANALYTICS_TABLES - tables.keys()
if missing:
    raise RuntimeError(f"The lakehouse is missing expected tables: {sorted(missing)}")

for name, table in sorted(tables.items()):
    selected = name in SUPPLIER_ANALYTICS_TABLES
    if table["isSelected"] != selected:
        fabric_call("PATCH", elements_url, params={"id": table["id"]}, json={"isSelected": selected})
    print(f"{'✓ selected  ' if selected else '  skipped   '} {name}")

## Step 6: Add instructions and publish the data agent

Instructions tell the data agent how the tables fit together and how to present the numbers. The agent-level instructions guide every answer, and the data source instructions describe the lakehouse. Publishing turns the draft into the version that the MCP endpoint serves.

The cell also saves the MCP URL to `.env` as `FABRIC_DATA_AGENT_MCP_URL`, so you can use it in the optional GitHub Copilot app activity.

In [ ]:
from dotenv import find_dotenv, set_key

AI_INSTRUCTIONS = """Answer Caldova supplier performance questions from the selected lakehouse tables.
Report OTIF12moPct, BatchRejectionRatePct, RightFirstTimePct, and CurrentUtilizationPct as percentages, since they are stored as fractions.
Performance metrics are synthetic analytics, not facts extracted from invoices. Treat null metrics as not applicable, never as zero.
Do not answer product, active-substance, or manufacturer relationship questions. Those come from the Caldova ontology."""

DATASOURCE_INSTRUCTIONS = """SupplierPerformance holds one weekly snapshot per supplier, keyed by SupplierID and DateKey.
Join DimSupplier on SupplierID for supplier names and DimDate on DateKey for snapshot dates.
Use the latest DateKey for current values, and order by DateKey for trends."""

fabric_call("PATCH", f"{DATA_AGENT_URL}/staging/settings", json={"aiInstructions": AI_INSTRUCTIONS})
fabric_call(
    "PATCH",
    f"{DATA_AGENT_URL}/staging/datasources/{datasource_id}",
    json={"description": "Caldova supplier performance star schema", "instructions": DATASOURCE_INSTRUCTIONS},
)
fabric_call(
    "POST",
    f"{DATA_AGENT_URL}/staging/publish",
    json={"publishedDescription": "Caldova supplier performance analytics"},
)

FABRIC_DATA_AGENT_MCP_URL = f"{FABRIC_API_URL}/v1/mcp/workspaces/{FABRIC_WORKSPACE_ID}/dataagents/{data_agent_id}/agent"
env_path = find_dotenv(usecwd=True)
if env_path:
    set_key(env_path, "FABRIC_DATA_AGENT_MCP_URL", FABRIC_DATA_AGENT_MCP_URL, quote_mode="never")
print(f"Published '{DATA_AGENT_NAME}'")
print(f"MCP endpoint: {FABRIC_DATA_AGENT_MCP_URL}")

## Step 7: Ask the data agent which suppliers are falling behind

Open an MCP session to the published data agent, find its tool, and ask an analytical question. The data agent writes a query over the tables you selected, runs it, and summarizes the result. The response also includes a deep link that opens the answer in Fabric, where you can inspect the generated query.

⏱️ Each answer takes about 20 to 40 seconds.

In [ ]:
async def ask_data_agent(question):
    async with open_fabric_mcp_session(FABRIC_DATA_AGENT_MCP_URL) as session:
        tools_response = await session.list_tools()
        tool_name = tools_response.tools[0].name
        result = await session.call_tool(tool_name, {"userQuestion": question})
    if result.isError:
        raise RuntimeError(f"{tool_name} failed: {result.content}")
    answer = "\n\n".join(block.text for block in result.content if block.type == "text")
    deep_link_url = (result.structuredContent or {}).get("deepLinkUrl")
    if deep_link_url:
        answer += f"\n\n[Open this answer in Fabric]({deep_link_url})"
    console.print(f"[bold cyan]Tool:[/] {tool_name}")
    console.print(Markdown(answer))


await ask_data_agent(
    "Which three suppliers have the lowest OTIF in the latest weekly snapshot, and what are their batch rejection rates?"
)

## Step 8: Follow Meridian API Works over time

Now ask about the supplier behind lot `AUR-API-7`. A trend needs several snapshots, which is the kind of aggregation the ontology does not do. Change the question to explore other suppliers or metrics.

In [ ]:
await ask_data_agent(
    "How did Meridian API Works' OTIF and batch rejection rate change over the last four weekly snapshots?"
)

#### 🔍 Source Hunt

Compare the answers from Steps 3, 7, and 8. Can you find:

1. Which answer came from **relationships** in the ontology, and which came from **aggregations** over the lakehouse tables?
2. Open a deep link from Step 7 or 8. Which tables does the generated query read, and does it use the latest `DateKey`?
3. In the lakehouse, preview `SupplierPerformance` for Meridian API Works. Do the latest values match the data agent's answer?
4. Ask the data agent which products contain Caldovexine. Does it decline, as its instructions require?

## ✅ Mission complete

**What you built:**

* ✓ **Ontology over MCP**: `ask_ontology` answered which manufacturers make Caldovexine and which products contain it, from the relationships modelled in `CaldovaMedicinalProductOntology`.
* ✓ **Fabric data agent**: `CaldovaSupplierDataAgent`, created and published with the Fabric REST API over the supplier analytics tables, with instructions for how the tables join and how to present the metrics.
* ✓ **Analytics over MCP**: The data agent answered ranking and trend questions about supplier performance, with a deep link to the generated query.

Fabric IQ gives agents structured business data. The next part adds what people are saying and deciding in Microsoft 365.

➡️ Select **Next** in the lab sidebar to start the [Exercise 3 browser exploration](../../instructions/overview.md#exercise-3-bring-in-work-context-with-work-iq), then open [Part 3: Bring in work context with Work IQ](part3-work-iq.ipynb). Outside Skillable, use the exercise link.